# Ejercicio: Predicción de Costes de Seguros Médicos (Insurance Dataset)

En este notebook trabajaremos con un dataset real de seguros médicos. El objetivo es predecir los gastos médicos de un paciente (`charges`) basándonos en sus características demográficas y hábitos de salud.

## Información del Dataset

El dataset contiene las siguientes variables:

| Variable | Tipo | Descripción |
| :--- | :--- | :--- |
| **age** | Numérica | Edad del beneficiario principal. |
| **sex** | Categórica | Género del contratante (male / female). |
| **bmi** | Numérica | Índice de masa corporal ($kg/m^2$). |
| **children** | Numérica | Número de hijos/dependientes cubiertos por el seguro. |
| **smoker** | Categórica | Indica si el beneficiario fuma (yes / no). |
| **region** | Categórica | Zona residencial en EE. UU. (northeast, southeast, southwest, northwest). |
| **charges** (Target) | Numérica | Gastos médicos individuales facturados por el seguro. |

---

# 1. Carga de Datos y Exploración Inicial

Primero, cargamos el dataset directamente desde el archivo y visualizamos su estructura.

In [11]:
import pandas as pd
import numpy as np

# Carga del dataset desde URL
df = pd.read_csv('insurance.csv')

# Mostramos una porción y el tamaño del dataset
print("Dimensiones del dataset:", df.shape)
display(df.head())

# Información sobre tipos de datos y nulos
print("\n-- Nulos y tipos --")
df.info()

Dimensiones del dataset: (1338, 7)


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520



-- Nulos y tipos --
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1338 entries, 0 to 1337
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       1338 non-null   int64  
 1   sex       1338 non-null   object 
 2   bmi       1338 non-null   float64
 3   children  1338 non-null   int64  
 4   smoker    1338 non-null   object 
 5   region    1338 non-null   object 
 6   charges   1338 non-null   float64
dtypes: float64(2), int64(2), object(3)
memory usage: 73.3+ KB


# 2. Transformación de variables categóricas

Los modelos que hemos estudiado hasta ahora (**OLS**, **SGD**, **Lasso**, **Ridge**) son modelos matemáticos basados en álgebra lineal. Funcionan calculando distancias, pendientes y productos matriciales.

> **Regla de oro:** Un modelo de regresión lineal solo entiende de números. No sabe qué significa "femenino" o "fumador", por lo que debemos traducir esas palabras a valores numéricos.

### ¿Existen modelos que trabajen con variables categóricas directamente?
Sí, existen modelos basados en **Árboles de Decisión** (como *Random Forest* o *Gradient Boosting*) y otras librerías que pueden gestionar variables de texto de forma nativa. Sin embargo, internamente, incluso estos modelos suelen realizar una conversión automática para poder operar.

Pero por ahora, necesitamos transformar las variables categóricas a numéricas.


## Estrategias de codificación (Encoding)

Para transformar nuestras variables de texto a números, utilizaremos dos estrategias principales dependiendo de cuántas categorías tenga la variable:

### A. Label Encoding (Codificación Binaria)
Se usa cuando la variable tiene solo **dos opciones** (binaria).
* **Ejemplo:** `smoker` (yes/no) o `sex` (male/female).
* **Resultado:** Se asigna un `0` a una categoría y un `1` a la otra. 
* **Ventaja:** No aumenta el número de columnas del dataset.

### B. One-Hot Encoding (Variables Dummy)
Se usa cuando la variable tiene **tres o más opciones** sin un orden jerárquico.
* **Ejemplo:** `region` (northeast, southeast, southwest, northwest).
* **Problema:** Si asignamos números 1, 2, 3 y 4, el modelo pensará que "northwest (4)" vale cuatro veces más que "northeast (1)", lo cual es un error conceptual.
* **Solución:** Creamos una columna nueva por cada categoría. Cada columna solo contendrá ceros y unos. Esto lo podemos crear con matriz Dummy de Pandas, pero surgen 2 problemas a tener en cuenta.
  
    1. **Redundancia:** Si tenemos dos columnas, `Fumador_Sí` y `Fumador_No`, la segunda no aporta información nueva. Si el valor en la primera es **0**, sabemos con 100% de certeza que en la segunda será **1**. 

    2. **Multicolinealidad de variables** Los modelos lineales fallan cuando una variable es una combinación perfecta de otras. Si dejas todas las columnas del Dummy, una columna está correlacionada con las otras 3. Esto genera coeficientes inestables y estimaciones poco fiables. 
     
   > **En conclusión:** Con $n-1$ columnas representas el 100% de la información sin redundancias y sin provocar errores en el modelo.

In [12]:
# Copiamos el dataframe para no tocar el original
df_encoded = df.copy()

# 1. Aplicamos Label Encoding usando .map() de Pandas (o replace) y un dict
df_encoded['smoker'] = df_encoded['smoker'].map({'no': 0, 'yes': 1})
df_encoded['sex'] = df_encoded['sex'].map({'female': 0, 'male': 1})


# 2. Aplicamos One-Hot Encoding a la variable 'region' con get_dummies
# drop_first=True elimina una de las columnas creadas para evitar la redundancia matemática
# dtype=int para que no salga True/False
df_encoded = pd.get_dummies(df_encoded, columns=['region'], drop_first=True, dtype=int)

df_encoded.head()

,age,sex,bmi,children,smoker,charges,region_northwest,region_southeast,region_southwest
0,19,0,27.900,0,1,16884.92400,0,0,1
1,18,1,33.770,1,0,1725.55230,0,1,0
2,28,1,33.000,3,0,4449.46200,0,1,0
3,33,1,22.705,0,0,21984.47061,1,0,0
4,32,1,28.880,0,0,3866.85520,1,0,0


---
# 3. Separación de Variables (X e y)

Antes de realizar cualquier transformación, separamos las características (**X**) de nuestra variable objetivo (**y**). 

En este punto, convertiremos los datos a **arrays de Numpy**, que es el formato estándar para trabajar con la mayoría de algoritmos de Scikit-Learn.

In [13]:
# Definimos la variable objetivo (Target)
y = df_encoded['charges'].values

# Definimos las características (Features)
# Eliminamos la columna charges de X
X = df_encoded.drop('charges', axis=1).values

print("Tipo de dato de X:", type(X))
print("Forma de X (filas, columnas):", X.shape)
print("\nPrimeras 3 filas de X:")
print(X[:3])

Tipo de dato de X: <class 'numpy.ndarray'>
Forma de X (filas, columnas): (1338, 8)

Primeras 3 filas de X:
[[19.    0.   27.9   0.    1.    0.    0.    1.  ]
 [18.    1.   33.77  1.    0.    0.    1.    0.  ]
 [28.    1.   33.    3.    0.    0.    1.    0.  ]]


# Tareas a realizar

Vamos a poner a prueba diferentes algoritmos. El objetivo no es solo "ejecutar código", sino **experimentar con los hiperparámetros** para encontrar el modelo que mejor generalice con datos que nunca ha visto.

### 4. División del Dataset
Para evaluar si nuestro modelo realmente aprende o solo memoriza, dividiremos los datos:
* **Entrenamiento (80%)**: Datos para que el modelo aprenda los pesos.
* **Test (20%)**: Datos "examen" para evaluar el rendimiento final.
* **Nota:** Usa `random_state=42` para garantizar que todos obtengamos los mismos resultados.

In [14]:
from sklearn.model_selection import train_test_split

# Dividimos en entrenamiento y test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (1070, 8)
Test: (268, 8)


### 5. Escalado de Datos
Los modelos lineales son sensibles a la magnitud de los números. 
1. Aplica un escalador (ej. **RobustScaler**) a los conjuntos de entrenamiento y test.
2. **Visualización:** Genera histogramas de las variables tras el escalado. ¿Qué ha cambiado en los ejes X?

In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### 6. Entrenamiento de Modelos (La Gran Comparativa)

Realiza el entrenamiento de cada modelo en celdas separadas. Tu misión es **ajustar los hiperparámetros** de cada algoritmo:

* **A. Regresión Lineal Clásica (OLS):** Úsalo como tu "Línea Base" (Base Model). Es el modelo más sencillo y no requiere ajuste de hiperparámetros.
* **B. Descenso de Gradiente Estocástico (SGDRegressor):** Este modelo es sensible. Prueba a variar la tasa de aprendizaje (`eta0`) y el tipo de `learning_rate`. 
    * *¿Consigues que converja sin que los errores se disparen al infinito?*
* **C. Regularización Lasso:** Experimenta con **3 valores distintos de `alpha`** (ej. 0.01, 0.1, 1).
    * Por cada valor, comprueba cuántos coeficientes han sido eliminados (reducidos a cero). ¿Qué variables considera Lasso que no aportan valor?
* **D. Regularización Ridge:** A diferencia de Lasso, Ridge no elimina variables, sino que distribuye el error entre todas ellas "encogiendo" sus coeficientes. Prueba con valores de `alpha` pequeños y grandes. 
    * ¿Cómo se comparan sus coeficientes con los de la Regresión Lineal clásica?
* **E. ElasticNet (híbrido):** Este modelo combina lo mejor de ambos mundos (Lasso y Ridge). Es ideal cuando tenemos muchas variables correlacionadas entre sí.
    * **Hiperparámetros clave:**
        * `alpha`: Controla la fuerza global de la penalización.
        * `l1_ratio`: Si es 1.0, se comporta como Lasso; si es 0.0, como Ridge. 
    * **Experimento:** Prueba un `l1_ratio=0.5` para ver cómo equilibra la selección de variables con la estabilidad.

In [16]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

lr = LinearRegression()
lr.fit(X_train_scaled, y_train)

y_pred_lr = lr.predict(X_test_scaled)

print("OLS")
print("MSE:", mean_squared_error(y_test, y_pred_lr))
print("R2:", r2_score(y_test, y_pred_lr))

OLS
MSE: 33596915.85136146
R2: 0.7835929767120723


In [17]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

y_pred_ridge = ridge.predict(X_test_scaled)

print("\nRidge")
print("MSE:", mean_squared_error(y_test, y_pred_ridge))
print("R2:", r2_score(y_test, y_pred_ridge))


Ridge
MSE: 33604973.5399633
R2: 0.7835410749121386


In [18]:
for a in [0.1, 1, 10, 100]:
    model = Ridge(alpha=a)
    model.fit(X_train_scaled, y_train)
    pred = model.predict(X_test_scaled)
    print(f"alpha={a} → R2:", r2_score(y_test, pred))

alpha=0.1 → R2: 0.7835878415412586
alpha=1 → R2: 0.7835410749121386
alpha=10 → R2: 0.7830200444609898
alpha=100 → R2: 0.7734197118072562


In [19]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, y_train)

y_pred_lasso = lasso.predict(X_test_scaled)

print("\nLasso")
print("MSE:", mean_squared_error(y_test, y_pred_lasso))
print("R2:", r2_score(y_test, y_pred_lasso))


Lasso
MSE: 33597337.55947545
R2: 0.7835902603735236


In [20]:
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(max_iter=1000, eta0=0.01, learning_rate='invscaling')
sgd.fit(X_train_scaled, y_train)

y_pred_sgd = sgd.predict(X_test_scaled)

print("\nSGD")
print("MSE:", mean_squared_error(y_test, y_pred_sgd))
print("R2:", r2_score(y_test, y_pred_sgd))


SGD
MSE: 33596688.80510007
R2: 0.7835944391798118


### 7. Resumen y comparación del Rendimiento
Para cada uno de tus mejores intentos en los modelos anteriores, calcula en el conjunto de **Test**:
1.  **R² Ajustado:** ¿Qué porcentaje de la varianza explicamos realmente?
2.  **MAE (Error Absoluto Medio):** ¿Cuánto fallamos de media en dólares?
3.  **RMSE (Raíz del Error Cuadrático Medio):** ¿Tenemos errores muy grandes que penalicen el modelo?

> **Elección:** Tras comparar las métricas, ¿qué modelo elegirías para poner en producción?

### 8. Persistencia
* Guarda el **mejor modelo** y el **escalador** en archivos `.pkl` utilizando la librería `joblib`.
* ¡Felicidades! Tu modelo está listo para ser integrado en una aplicación real.

In [21]:
results = {
    "OLS": r2_score(y_test, y_pred_lr),
    "Ridge": r2_score(y_test, y_pred_ridge),
    "Lasso": r2_score(y_test, y_pred_lasso),
    "SGD": r2_score(y_test, y_pred_sgd),
}

print("\nComparación de modelos (R2):")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


Comparación de modelos (R2):
OLS: 0.7836
Ridge: 0.7835
Lasso: 0.7836
SGD: 0.7836
